# Remesh rule and field transfer

After the model predicts the last frame of an epoch (frame 10e+10 on mesh e), the remeshing step builds the next mesh from that predicted phase field alone and moves fields onto it. Code: `amr/remesh.py`.

**Rule.** Mark every cell that lies within distance `d` of the phi = 0 interface; marked cells refine one level, groups of four unmarked siblings coarsen one level; then 2:1 balance across faces and corners. The solver's own rule (any nodal phi in (-0.9999, 0.9999)) reproduces the solver's meshes on true fields, but reacts to small bulk errors of a predicted field; the distance rule depends only on where the interface is. For phi = tanh(n / sqrt(2)) the solver band reaches about 7 length units on each side, so `d` near 7 imitates the solver.

**Perturbations that imitate model errors** (applied to the true field before marking): `none`; `bulk_bias` (phi + 0.01 g, g a smooth random field in [-1, 1]); `shift_out` / `shift_in` (interface moved 1.5 length units into the liquid / solid).

**Measures per transition e -> e+1:** `area_same_level` (fraction of the domain with the solver's cell size), `coverage_start` / `coverage_end` (fraction of the true phi = 0 points at the start / end of the next epoch that lie in finest cells: the safety measure), `cell_ratio` (cells / solver cells: the cost).

**Selection** on 20 training cases: a distance is safe if, for every perturbation, its smallest `coverage_start` is at least 0.999 and its mean `coverage_end` is at least that of the solver's mesh. Among safe distances, the best mean `area_same_level` under its worst perturbation wins. All 10 validation cases are then reported, followed by the mesh-build and field-transfer checks.

Run through HTCondor, not on the access point.

In [ ]:
cache_dir = 'cache'
split_path = 'splits.json'
out_dir = 'outputs/remesh'
train_stride = 5
distances = [4.0, 6.0, 7.0, 8.0, 10.0]
perturbations = ['none', 'bulk_bias', 'shift_out', 'shift_in']
bias_amplitude = 0.01
shift_length = 1.5
seed = 539
workers = 8

In [ ]:
import json
import os
import sys
import time
import zlib
from multiprocessing import Pool

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, os.getcwd())
from amr.remesh import (DISTANCE, build_mesh, compare, distance_marking, enthalpy, finest_coverage, leaves,
                      marking, remesh, transfer_matrix, transfer_state)
from amr.data import N_EPOCHS, interface_points, load_epoch, load_splits

os.makedirs(out_dir, exist_ok=True)
pd.set_option('display.width', 200)
split = load_splits(split_path)
select_cases = split['train'][::train_stride]
all_cases = split['train'] + split['val'] + split['test'] + split['outer']
print(len(select_cases), 'training cases for selection:', select_cases)

In [ ]:
def smooth_field(pos, rng, modes=4):
    """Sum of a few plane waves with wavelengths 50-250, scaled to max |g| = 1."""
    g = np.zeros(len(pos))
    for _ in range(modes):
        angle, wavelength, phase = rng.uniform(0, np.pi), rng.uniform(50, 250), rng.uniform(0, 2 * np.pi)
        k = 2 * np.pi / wavelength * np.array([np.cos(angle), np.sin(angle)])
        g += np.cos(pos @ k + phase)
    return g / np.abs(g).max()


def perturb(phi, pos, kind, rng):
    if kind == 'none':
        return phi
    if kind == 'bulk_bias':
        return phi + bias_amplitude * smooth_field(pos, rng)
    # phi = tanh(n / sqrt(2)) with n the signed distance into the solid; adding s to n moves phi = 0 by s.
    s = shift_length if kind == 'shift_out' else -shift_length
    return np.tanh(np.arctanh(np.clip(phi, -0.999999, 0.999999)) + s / np.sqrt(2))


def rule_flags(rule, now, d0, phi, points):
    if rule == 'solver_0.9999':
        return marking(d0['cells'], phi, 0.9999)
    return distance_marking(now, points, float(rule.split('_')[1]))


def run_transition(job):
    """All rules and perturbations for one transition e -> e+1 of one case."""
    case, e, rules = job
    d0, d1 = load_epoch(cache_dir, case, e), load_epoch(cache_dir, case, e + 1)
    now, target = leaves(d0), leaves(d1)
    start_pts = interface_points(d1['pos'], d1['edge_index'], d1['phi'][0])
    end_pts = interface_points(d1['pos'], d1['edge_index'], d1['phi'][10])
    base = {'case': case, 'epoch': e}
    rows = [{**base, 'perturbation': 'none', 'rule': 'solver_mesh', 'area_same_level': 1.0, 'cell_ratio': 1.0,
             'coverage_start': finest_coverage(target, start_pts), 'coverage_end': finest_coverage(target, end_pts)}]
    rng = np.random.default_rng([seed, zlib.crc32(case.encode()), e])
    pos = d0['pos'].astype(np.float64)
    for kind in perturbations:
        phi = perturb(d0['phi'][10].astype(np.float64), pos, kind, rng)
        points = interface_points(pos, d0['edge_index'], phi)
        for rule in rules:
            candidate, _ = remesh(now, rule_flags(rule, now, d0, phi, points))
            c = compare(candidate, target)
            rows.append({**base, 'perturbation': kind, 'rule': rule, 'area_same_level': c['area_same_level'],
                         'cell_ratio': c['predicted_cells'] / c['true_cells'],
                         'coverage_start': finest_coverage(candidate, start_pts),
                         'coverage_end': finest_coverage(candidate, end_pts)})
    return rows


def run_all(cases, rules, label):
    jobs = [(c, e, rules) for c in cases for e in range(N_EPOCHS - 1)]
    rows, start = [], time.time()
    with Pool(workers) as pool:
        for i, result in enumerate(pool.imap_unordered(run_transition, jobs), 1):
            rows += result
            if i % 100 == 0 or i == len(jobs):
                print(f'{label}: {i}/{len(jobs)} transitions, {time.time() - start:.0f} s', flush=True)
    return pd.DataFrame(rows)


def table(rows):
    r = rows[rows['rule'] != 'solver_mesh']
    return r.groupby(['rule', 'perturbation']).agg(
        area_mean=('area_same_level', 'mean'), coverage_start_min=('coverage_start', 'min'),
        coverage_end_mean=('coverage_end', 'mean'), cell_ratio_mean=('cell_ratio', 'mean')).reset_index()

## 1. Choose the distance on training cases

In [ ]:
distance_rules = [f'distance_{d:g}' for d in distances]
train_rows = run_all(select_cases, ['solver_0.9999'] + distance_rules, 'train')
train_table = table(train_rows)
solver_coverage_end = float(train_rows.loc[train_rows['rule'] == 'solver_mesh', 'coverage_end'].mean())
print(train_table.to_string(index=False))

rule_table = train_table.groupby('rule').agg(
    worst_perturbation_area_mean=('area_mean', 'min'), coverage_start_min=('coverage_start_min', 'min'),
    coverage_end_mean_lowest=('coverage_end_mean', 'min'), cell_ratio_mean_highest=('cell_ratio_mean', 'max')).reset_index()
rule_table['safe'] = (rule_table['coverage_start_min'] >= 0.999) & (rule_table['coverage_end_mean_lowest'] >= solver_coverage_end)
rule_table = rule_table.sort_values(['safe', 'worst_perturbation_area_mean'], ascending=False)
rule_table.to_csv(os.path.join(out_dir, 'train_rule_table.csv'), index=False)
print('solver mesh mean coverage_end: %.5f' % solver_coverage_end)
print(rule_table.to_string(index=False))

candidates = rule_table[rule_table['rule'].isin(distance_rules)]
safe = candidates[candidates['safe']]
selected = (safe if len(safe) else candidates.sort_values('coverage_end_mean_lowest', ascending=False)).iloc[0]['rule']
selected_distance = float(selected.split('_')[1])
print('selected:', selected, '| distance used by amr.remesh:', DISTANCE)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for kind in perturbations:
    g = train_table[train_table['rule'].isin(distance_rules) & (train_table['perturbation'] == kind)].copy()
    g['d'] = g['rule'].str.split('_').str[1].astype(float)
    g = g.sort_values('d')
    for ax, col in zip(axes, ['area_mean', 'coverage_end_mean', 'cell_ratio_mean']):
        ax.plot(g['d'], g[col], '-o', label=kind)
axes[1].axhline(solver_coverage_end, color='black', lw=0.8, ls='--', label='solver mesh')
for ax, title in zip(axes, ['mean area with the solver cell size', 'mean interface coverage at epoch end',
                            'mean cells / solver cells']):
    ax.set_title(title)
    ax.set_xlabel('distance d')
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(out_dir, 'distance_selection.png'), dpi=120)

## 2. Validation: all 10 cases, all 29 transitions

In [ ]:
val_rows = run_all(split['val'], [f'distance_{DISTANCE:g}', 'solver_0.9999'], 'val')
val_table = table(val_rows)
val_solver = val_rows[val_rows['rule'] == 'solver_mesh']
print('solver mesh: coverage_start min %.5f, coverage_end mean %.5f'
      % (val_solver['coverage_start'].min(), val_solver['coverage_end'].mean()))
print(val_table.to_string(index=False))

## 3. Mesh build and field transfer

1. **Mesh build.** Rebuild every case's mesh at the first and last epoch from its quadtree leaves; nodes (matched by coordinates), node types, graph edges and integration weights must equal the solver's.
2. **Transfer onto the solver's next mesh** (10 validation cases, 29 transitions): frame 10e+10 on mesh e moved onto mesh e+1 and compared with the solver's own first frame of epoch e+1.
3. **Transfer onto the new mesh built by remeshing**: same, with the reference being the solver's frame evaluated exactly at the new nodes.

Enthalpy H = integral of (u - phi/2) and solid-area changes are divided by the latent heat of the current solid (0.5 x solid area).

In [ ]:
def coordinate_keys(pos):
    return [tuple(k) for k in np.rint(np.asarray(pos, dtype=np.float64) * 1e4).astype(np.int64)]


def check_build(job):
    case, e = job
    d = load_epoch(cache_dir, case, e)
    m = build_mesh(leaves(d), d['params'])
    solver_index = {k: i for i, k in enumerate(coordinate_keys(d['pos']))}
    matched = [solver_index.get(k, -1) for k in coordinate_keys(m['pos'])]
    row = {'case': case, 'epoch': e, 'unmatched_nodes': int(sum(i < 0 for i in matched)),
           'same_node_count': len(m['pos']) == len(d['pos'])}
    if row['unmatched_nodes'] == 0 and row['same_node_count']:
        perm = np.array(matched)
        row['node_type_equal'] = bool((m['node_type'] == d['node_type'][perm]).all())
        row['weight_rel_error'] = float(np.max(np.abs(m['weights'] - d['weights'][perm]) / d['weights'][perm]))
        row['edges_equal'] = {(perm[a], perm[b]) for a, b in m['edge_index'].T} == set(map(tuple, d['edge_index'].T))
    return row


def check_transition(job):
    case, e = job
    d0, d1 = load_epoch(cache_dir, case, e), load_epoch(cache_dir, case, e + 1)
    delta = float(d0['params'][0])
    phi0, u0 = d0['phi'][10].astype(np.float64), d0['u'][10].astype(np.float64)
    state = np.column_stack([phi0, u0])
    phi_ref, u_ref = d1['phi'][0].astype(np.float64), d1['u'][0].astype(np.float64)
    now = leaves(d0)
    points = interface_points(d0['pos'].astype(np.float64), d0['edge_index'], phi0)
    cells, _ = remesh(now, distance_marking(now, points))
    own = build_mesh(cells, d0['params'])
    targets = {'solver_mesh': (d1, phi_ref, u_ref)}
    ref_own = transfer_matrix(d1, own['pos']) @ np.column_stack([phi_ref, u_ref])
    targets['own_mesh'] = (own, ref_own[:, 0], ref_own[:, 1])
    solid = (d0['weights'] * (1 + phi0) / 2).sum()
    row = {'case': case, 'epoch': e, 'own_nodes_over_solver': len(own['pos']) / len(d1['pos'])}
    for name, (mesh, p_ref, u_r) in targets.items():
        t0 = time.perf_counter()
        moved = transfer_state(d0, state, mesh, delta)
        w = mesh['weights']
        row.update({f'{name}_seconds': time.perf_counter() - t0,
                    f'{name}_phi_max_error': float(np.abs(moved[:, 0] - p_ref).max()),
                    f'{name}_u_max_error_over_delta': float(np.abs(moved[:, 1] - u_r).max() / delta),
                    f'{name}_enthalpy_change': (enthalpy(w, moved[:, 0], moved[:, 1])
                                                - enthalpy(d0['weights'], phi0, u0)) / (0.5 * solid),
                    f'{name}_solid_area_change': ((w * (1 + moved[:, 0]) / 2).sum() - solid) / solid})
    return row


with Pool(workers) as pool:
    build = pd.DataFrame(pool.map(check_build, [(c, e) for c in all_cases for e in (0, N_EPOCHS - 1)]))
    transfer = pd.DataFrame(pool.map(check_transition, [(c, e) for c in split['val'] for e in range(N_EPOCHS - 1)]))
build.to_csv(os.path.join(out_dir, 'mesh_build_check.csv'), index=False)
transfer.to_csv(os.path.join(out_dir, 'transfer_check.csv'), index=False)
cols = [c for c in transfer.columns if c not in ('case', 'epoch')]
print(transfer[cols].abs().describe().T[['50%', 'max']])

In [ ]:
summary = {
    'selected_distance': selected_distance, 'distance_in_amr': DISTANCE,
    'train_rule_table': rule_table.to_dict(orient='records'),
    'val_table': val_table.to_dict(orient='records'),
    'mesh_build': {'meshes': int(len(build)),
                   'all_equal': bool((build['unmatched_nodes'] == 0).all() and build['same_node_count'].all()
                                     and build['node_type_equal'].all() and build['edges_equal'].all()),
                   'weight_rel_error_max': float(build['weight_rel_error'].max())},
    'transfer_max_abs': {c: float(transfer[c].abs().max()) for c in cols if 'seconds' not in c},
    'transfer_seconds_median': {c: float(transfer[c].median()) for c in cols if 'seconds' in c}}
with open(os.path.join(out_dir, 'summary.json'), 'w') as fh:
    json.dump(summary, fh, indent=1)
print(json.dumps({k: v for k, v in summary.items() if 'table' not in k}, indent=1))
assert summary['mesh_build']['all_equal']